# DermoGraph-XAI — VGG16 Baseline
**Team 8 | VIT Bhopal | Skin Lesion Classification**

| | |
|---|---|
| Model | VGG16 (ImageNet pretrained) |
| Dataset | HAM10000 + PAD-UFES-20 |
| Classes | 7 (Melanoma, Nevi, BCC, AK, BKL, DF, Vasc) |
| Loss | Cross Entropy + Class Weights |
| Optimizer | AdamW + CosineAnnealingLR |


## Cell 1 — Install & Verify GPU

In [ ]:
!pip install -q timm albumentations

import torch
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"PyTorch : {torch.__version__}")
print("✓ GPU ready" if torch.cuda.is_available() else "✗ NO GPU — go to Settings > Accelerator > GPU P100")


## Cell 2 — Imports & Config

In [ ]:
import os, json, time, warnings
warnings.filterwarnings('ignore')
os.environ['NO_ALBUMENTATIONS_UPDATE'] = '1'

import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm

from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              confusion_matrix, classification_report)

# ── CONFIG ────────────────────────────────────────────────────
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT      = '/kaggle/working/'
CACHE_DIR   = '/kaggle/working/cache/'
BATCH_SIZE  = 32
N_CLASSES   = 7
N_EPOCHS    = 50
LR          = 1e-4
PATIENCE    = 15
GRAD_ACCUM  = 4
MODEL_NAME  = 'vgg16'
MEAN        = [0.485, 0.456, 0.406]
STD         = [0.229, 0.224, 0.225]
CLASS_NAMES = ['Melanoma','Nevi','Basal Cell Carcinoma',
               'Actinic Keratosis','Benign Keratosis',
               'Dermatofibroma','Vascular']

torch.manual_seed(42)
np.random.seed(42)
os.makedirs(OUTPUT,    exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

print("✓ Config ready")
print(f"  Device  : {DEVICE}")
print(f"  Classes : {N_CLASSES}")


## Cell 3 — Load Dataset & Remap Paths

In [ ]:
# ── Dataset paths (your Kaggle datasets) ─────────────────────
HAM1   = '/kaggle/input/datasets/akshat23029/dermograph-ham-images/HAM10000_images_part_1'
HAM2   = '/kaggle/input/datasets/akshat23029/dermograph-ham-images/HAM10000_images_part_2'
PAD    = '/kaggle/input/datasets/akshat23029/dermograph-pad-images/images'
SPLITS = '/kaggle/input/datasets/akshat23029/dermograph-splits'

# Verify all paths exist
print("Checking dataset paths...")
ok = True
for name, path in [('HAM part1',HAM1),('HAM part2',HAM2),('PAD images',PAD),('Splits',SPLITS)]:
    exists = os.path.exists(path)
    n      = len(os.listdir(path)) if exists else 0
    print(f"  {'✓' if exists else '✗'} {name}: {n} files")
    if not exists: ok = False
assert ok, "Missing datasets! Add them via + Add Input in right panel"

# Load splits
train_df = pd.read_csv(f'{SPLITS}/train.csv')
val_df   = pd.read_csv(f'{SPLITS}/val.csv')
test_df  = pd.read_csv(f'{SPLITS}/test.csv')
with open(f'{SPLITS}/class_weights.json') as f:
    cw_raw = json.load(f)

# Build PAD-UFES image lookup
pad_lookup = {}
for part in ['imgs_part_1','imgs_part_2','imgs_part_3']:
    d = f"{PAD}/{part}"
    if not os.path.exists(d): continue
    for fname in os.listdir(d):
        pad_lookup[fname] = f"{d}/{fname}"
print(f"\nPAD-UFES lookup: {len(pad_lookup):,} images")

# Remap Mac paths → Kaggle paths
def remap(mac_path):
    p     = str(mac_path)
    fname = os.path.basename(p)
    if 'HAM10000_images_part_1' in p:
        fp = f"{HAM1}/{fname}"
        return fp if os.path.exists(fp) else None
    if 'HAM10000_images_part_2' in p:
        fp = f"{HAM2}/{fname}"
        return fp if os.path.exists(fp) else None
    if 'PAD-UFES' in p or 'imgs_part' in p:
        return pad_lookup.get(fname)
    return None

# Apply remap + remove 'Other' class (label=7, only 24 samples)
print("\nRemapping paths...")
for name, df in [('train',train_df),('val',val_df),('test',test_df)]:
    df['kpath'] = df['image_path'].apply(remap)
    df.drop(df[df['kpath'].isna()].index, inplace=True)
    df.drop(df[df['label'] >= N_CLASSES].index, inplace=True)
    df.reset_index(drop=True, inplace=True)
    src = df['source'].value_counts().to_dict()
    print(f"  {name}: {len(df):,} images | {src}")

print("\n✓ Data loaded and ready")


## Cell 4 — Cache Preprocessed Images (Runs Once — Makes Training 3x Faster)

In [ ]:
def remove_hair(img_bgr):
    gray     = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    kernel   = cv2.getStructuringElement(cv2.MORPH_RECT, (11,11))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    _, mask  = cv2.threshold(blackhat, 10, 255, cv2.THRESH_BINARY)
    dk       = cv2.getStructuringElement(cv2.MORPH_RECT, (3,3))
    mask     = cv2.dilate(mask, dk, iterations=1)
    return cv2.inpaint(img_bgr, mask, 3, cv2.INPAINT_TELEA)

# Combine all dataframes to cache everything at once
all_paths = pd.concat([
    train_df[['kpath']],
    val_df[['kpath']],
    test_df[['kpath']]
], ignore_index=True)['kpath'].tolist()

unique_paths = list(set(all_paths))
print(f"Caching {len(unique_paths):,} unique images...")
print("This runs ONCE — hair removal pre-applied, 224x224 saved")
print("After this each epoch takes ~3-4 min instead of 15 min")

done = skipped = errors = 0
for src_path in tqdm(unique_paths, desc="Preprocessing"):
    fname    = os.path.basename(src_path)
    dst_path = f"{CACHE_DIR}{fname}"
    if os.path.exists(dst_path):
        skipped += 1
        continue
    img = cv2.imread(src_path)
    if img is None:
        errors += 1
        continue
    img = remove_hair(img)
    img = cv2.resize(img, (224,224), interpolation=cv2.INTER_LINEAR)
    cv2.imwrite(dst_path, img)
    done += 1

print(f"\n✓ Cache complete")
print(f"  Processed : {done:,}")
print(f"  Skipped   : {skipped:,} (already cached)")
print(f"  Errors    : {errors}")

# Update paths to use cache
def to_cache(p):
    fname = os.path.basename(str(p))
    cp    = f"{CACHE_DIR}{fname}"
    return cp if os.path.exists(cp) else str(p)

for df in [train_df, val_df, test_df]:
    df['kpath'] = df['kpath'].apply(to_cache)

print(f"  Cache size: {sum(os.path.getsize(CACHE_DIR+f) for f in os.listdir(CACHE_DIR))/1e9:.2f} GB")
print("✓ All paths updated to use cache")


## Cell 5 — Dataset Class & Augmentation

In [ ]:
def get_train_transforms():
    return A.Compose([
        A.RandomRotate90(p=0.5),
        A.Rotate(limit=180, p=0.7),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
        A.GaussianBlur(blur_limit=(3,7), p=0.3),
        A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.3),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

def get_val_transforms():
    return A.Compose([
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

class DermDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        img   = cv2.imread(str(row['kpath']))
        if img is None:
            img = np.zeros((224,224,3), dtype=np.uint8)
        img   = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(image=img)['image']
        label = torch.tensor(int(row['label']), dtype=torch.long)
        return img, label

# Quick test
ds    = DermDataset(train_df, get_train_transforms())
img, lbl = ds[0]
print(f"✓ Dataset test:")
print(f"  Image shape : {img.shape}  (C x H x W)")
print(f"  Image dtype : {img.dtype}")
print(f"  Label       : {lbl.item()} = {CLASS_NAMES[lbl.item()]}")
print(f"  Train size  : {len(ds):,}")


## Cell 6 — DataLoaders & Class Weights

In [ ]:
# Simple shuffle — no WeightedRandomSampler (was causing instability)
train_ds = DermDataset(train_df, get_train_transforms())
val_ds   = DermDataset(val_df,   get_val_transforms())
test_ds  = DermDataset(test_df,  get_val_transforms())

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

# Class weights capped at 10.0 (189.99 was destroying training)
cw_list = [min(float(cw_raw.get(str(i), 1.0)), 10.0) for i in range(N_CLASSES)]
CLASS_WEIGHTS = torch.tensor(cw_list, dtype=torch.float32).to(DEVICE)

print("✓ DataLoaders ready")
print(f"  Train : {len(train_loader):>4} batches  ({len(train_ds):,} images)")
print(f"  Val   : {len(val_loader):>4} batches  ({len(val_ds):,} images)")
print(f"  Test  : {len(test_loader):>4} batches  ({len(test_ds):,} images)")
print(f"\nClass weights (capped at 10.0):")
for i, (name, w) in enumerate(zip(CLASS_NAMES, cw_list)):
    print(f"  {i} {name:<25} {w:.4f}")


## Cell 7 — Model, Loss & Optimizer

In [ ]:
# VGG16 with pretrained ImageNet weights
model    = timm.create_model(MODEL_NAME, pretrained=True, num_classes=N_CLASSES).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters()) / 1e6

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
scheduler = CosineAnnealingLR(optimizer, T_max=N_EPOCHS, eta_min=1e-6)
scaler    = GradScaler()

def criterion(logits, labels):
    """Weighted cross-entropy loss"""
    return F.cross_entropy(logits, labels, weight=CLASS_WEIGHTS)

def run_eval(model, loader):
    """Full evaluation — returns loss, acc, f1, auc, preds, labels, probs"""
    model.eval()
    t_loss = correct = total = 0
    preds_all, labels_all, probs_all = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            with autocast():
                logits = model(imgs)
                loss   = criterion(logits, labels)
            probs = torch.softmax(logits, 1)
            t_loss  += loss.item()
            correct += (logits.argmax(1) == labels).sum().item()
            total   += labels.size(0)
            preds_all.extend(logits.argmax(1).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
            probs_all.extend(probs.cpu().numpy())
    acc = correct / total
    f1  = f1_score(labels_all, preds_all, average='macro', zero_division=0)
    try:    auc = roc_auc_score(labels_all, probs_all, multi_class='ovr', average='macro')
    except: auc = 0.0
    return (t_loss/len(loader), acc, f1, auc,
            np.array(preds_all), np.array(labels_all), np.array(probs_all))

print(f"✓ VGG16 loaded")
print(f"  Parameters    : {n_params:.1f}M")
print(f"  Loss          : Weighted CrossEntropy (max_weight=10.0)")
print(f"  Optimizer     : AdamW  (lr={LR}, wd=1e-2)")
print(f"  Scheduler     : CosineAnnealingLR (T_max={N_EPOCHS})")
print(f"  Grad accum    : {GRAD_ACCUM}x → effective batch = {BATCH_SIZE*GRAD_ACCUM}")


## Cell 8 — Training (Saves Checkpoint Every Epoch)

In [ ]:
best_acc = best_f1 = best_auc = 0.0
patience = 0
history  = {'epoch':[],'train_loss':[],'val_loss':[],
            'val_acc':[],'val_f1':[],'val_auc':[]}
start    = time.time()

print(f"Training VGG16 — {N_EPOCHS} epochs max, early stop patience={PATIENCE}")
print("="*85)
print(f"{'Ep':>3} | {'TrainLoss':>9} | {'ValLoss':>8} | {'ValAcc':>7} | {'F1':>7} | {'AUC':>7} | {'Min':>5} | Status")
print("="*85)

for epoch in range(N_EPOCHS):
    # ── Train one epoch ───────────────────────────────────────────
    model.train()
    train_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(train_loader, desc=f"Ep {epoch+1:02d}/{N_EPOCHS} [Train]", leave=False)
    for step, (imgs, labels) in enumerate(pbar):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast():
            logits = model(imgs)
            loss   = criterion(logits, labels) / GRAD_ACCUM
        scaler.scale(loss).backward()
        if (step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        train_loss += loss.item() * GRAD_ACCUM
        pbar.set_postfix({'loss': f'{loss.item()*GRAD_ACCUM:.3f}'})
    tr_loss = train_loss / len(train_loader)

    # ── Validate ──────────────────────────────────────────────────
    va_loss, va_acc, va_f1, va_auc, _, _, _ = run_eval(model, val_loader)
    scheduler.step()

    elapsed = (time.time() - start) / 60
    status  = ''

    # Save checkpoint every epoch
    torch.save({
        'epoch':       epoch,
        'model_state': model.state_dict(),
        'optimizer':   optimizer.state_dict(),
        'val_acc':     va_acc,
        'val_f1':      va_f1,
        'val_auc':     va_auc,
    }, f'{OUTPUT}{MODEL_NAME}_ep{epoch:02d}.pth')

    # Save best model separately
    if va_acc > best_acc:
        best_acc = va_acc
        best_f1  = va_f1
        best_auc = va_auc
        patience = 0
        torch.save(model.state_dict(), f'{OUTPUT}{MODEL_NAME}_best.pth')
        status = '⭐ BEST'
    else:
        patience += 1
        status = f'patience {patience}/{PATIENCE}'
        if patience >= PATIENCE:
            status = '⛔ EARLY STOP'

    # Log
    history['epoch'].append(epoch)
    history['train_loss'].append(round(tr_loss, 4))
    history['val_loss'].append(round(va_loss, 4))
    history['val_acc'].append(round(va_acc, 4))
    history['val_f1'].append(round(va_f1, 4))
    history['val_auc'].append(round(va_auc, 4))

    print(f"{epoch+1:>3} | {tr_loss:>9.4f} | {va_loss:>8.4f} | {va_acc:>7.4f} | "
          f"{va_f1:>7.4f} | {va_auc:>7.4f} | {elapsed:>5.1f} | {status}")

    # Save history after every epoch (safe in case session dies)
    with open(f'{OUTPUT}{MODEL_NAME}_history.json', 'w') as f:
        json.dump(history, f, indent=2)

    if patience >= PATIENCE:
        break

total_min = (time.time()-start)/60
print("="*85)
print(f"\n✓ Training complete in {total_min:.1f} min")
print(f"  Best Val Acc  : {best_acc*100:.2f}%")
print(f"  Best Val F1   : {best_f1:.4f}")
print(f"  Best Val AUC  : {best_auc:.4f}")
print(f"  Epochs run    : {len(history['epoch'])}")


## Cell 9 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18,5))
fig.patch.set_facecolor('#0a0e1a')
for ax in axes:
    ax.set_facecolor('#0f1525')
    ax.spines[['top','right','left','bottom']].set_color('#1e2d4a')
    ax.tick_params(colors='#94a3b8')
    ax.set_xlabel('Epoch', color='#94a3b8')

ep = history['epoch']

axes[0].plot(ep, history['train_loss'], '#FF4444', lw=2, label='Train')
axes[0].plot(ep, history['val_loss'],   '#00e5cc', lw=2, label='Val')
axes[0].set_title('Loss', color='white', fontsize=13, fontweight='bold')
axes[0].legend(facecolor='#0f1525', edgecolor='#1e2d4a', labelcolor='white')

axes[1].plot(ep, history['val_acc'], '#00e5cc', lw=2, label='Val Accuracy')
axes[1].axhline(best_acc, color='#ffc849', ls='--', lw=1.5, label=f'Best {best_acc*100:.2f}%')
axes[1].set_title('Accuracy', color='white', fontsize=13, fontweight='bold')
axes[1].set_ylim(0,1)
axes[1].legend(facecolor='#0f1525', edgecolor='#1e2d4a', labelcolor='white')

axes[2].plot(ep, history['val_f1'],  '#a855f7', lw=2, label='F1 Macro')
axes[2].plot(ep, history['val_auc'], '#22c55e', lw=2, label='AUC-ROC')
axes[2].set_title('F1 + AUC', color='white', fontsize=13, fontweight='bold')
axes[2].set_ylim(0,1)
axes[2].legend(facecolor='#0f1525', edgecolor='#1e2d4a', labelcolor='white')

plt.suptitle(f'VGG16 Training — Best Val Acc={best_acc*100:.2f}%  AUC={best_auc:.4f}',
             color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT}vgg16_curves.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()
print("✓ Saved: vgg16_curves.png")


## Cell 10 — Test Evaluation

In [ ]:
print("Loading best model weights...")
model.load_state_dict(torch.load(f'{OUTPUT}{MODEL_NAME}_best.pth', map_location=DEVICE))

_, test_acc, test_f1, test_auc, all_preds, all_labels, all_probs = run_eval(model, test_loader)

print("\n" + "="*55)
print("  VGG16 — FINAL TEST RESULTS")
print("="*55)
print(f"  Accuracy  : {test_acc*100:.2f}%")
print(f"  F1 Macro  : {test_f1:.4f}")
print(f"  AUC-ROC   : {test_auc:.4f}")
print(f"  Val Acc   : {best_acc*100:.2f}%  (best during training)")
print(f"  Val AUC   : {best_auc:.4f}")
print("="*55)

present = sorted(set(all_labels))
names   = [CLASS_NAMES[i] for i in present]
print("\nPer-class Report:")
print(classification_report(all_labels, all_preds,
                             labels=present, target_names=names, zero_division=0))


## Cell 11 — Confusion Matrix

In [ ]:
present = sorted(set(all_labels))
names   = [CLASS_NAMES[i] for i in present]
cm      = confusion_matrix(all_labels, all_preds, labels=present)
cm_pct  = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(11,9))
fig.patch.set_facecolor('#0a0e1a')
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='YlOrRd',
            xticklabels=names, yticklabels=names, ax=ax,
            linewidths=0.5, linecolor='#1e2d4a',
            annot_kws={'size':10,'weight':'bold'})
ax.set_facecolor('#0f1525')
ax.set_xlabel('Predicted', color='white', fontsize=12)
ax.set_ylabel('Actual',    color='white', fontsize=12)
ax.set_title(f'VGG16 — Confusion Matrix (%)\nAcc={test_acc*100:.2f}%  F1={test_f1:.4f}  AUC={test_auc:.4f}',
             color='white', fontsize=13, fontweight='bold', pad=15)
plt.setp(ax.get_xticklabels(), rotation=35, ha='right', color='white', fontsize=9)
plt.setp(ax.get_yticklabels(), rotation=0,  color='white', fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUTPUT}vgg16_confusion_matrix.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()
print("✓ Saved: vgg16_confusion_matrix.png")


## Cell 12 — Per-Class Accuracy

In [ ]:
per_class_acc = cm.diagonal() / cm.sum(axis=1) * 100

fig, ax = plt.subplots(figsize=(12,5))
fig.patch.set_facecolor('#0a0e1a')
ax.set_facecolor('#0f1525')
colors = ['#FF4444' if a<70 else '#ffc849' if a<85 else '#22c55e' for a in per_class_acc]
bars = ax.bar(names, per_class_acc, color=colors, edgecolor='white', linewidth=0.5, width=0.6)
ax.axhline(test_acc*100, color='#00e5cc', ls='--', lw=2, label=f'Overall = {test_acc*100:.2f}%')
ax.set_title('VGG16 — Per-Class Accuracy', color='white', fontsize=13, fontweight='bold')
ax.set_ylabel('Accuracy (%)', color='#94a3b8')
ax.set_ylim(0, 115)
ax.tick_params(colors='#94a3b8')
ax.spines[['top','right','left','bottom']].set_color('#1e2d4a')
plt.setp(ax.get_xticklabels(), rotation=35, ha='right', color='white', fontsize=9)
for bar, val in zip(bars, per_class_acc):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1.5,
            f'{val:.1f}%', ha='center', color='white', fontsize=9, fontweight='bold')
patches = [mpatches.Patch(color='#22c55e', label='≥85% Good'),
           mpatches.Patch(color='#ffc849', label='70-85% OK'),
           mpatches.Patch(color='#FF4444', label='<70% Poor')]
ax.legend(handles=patches+[plt.Line2D([0],[0],color='#00e5cc',ls='--',lw=2,
          label=f'Overall {test_acc*100:.2f}%')],
          facecolor='#0f1525', edgecolor='#1e2d4a', labelcolor='white')
plt.tight_layout()
plt.savefig(f'{OUTPUT}vgg16_per_class_acc.png', dpi=150, bbox_inches='tight', facecolor='#0a0e1a')
plt.show()
print("✓ Saved: vgg16_per_class_acc.png")


## Cell 13 — Save All Results

In [ ]:
result = {
    'model':        'VGG16',
    'type':         'baseline',
    'params_M':     round(n_params, 1),
    'best_val_acc': round(best_acc, 4),
    'best_val_f1':  round(best_f1,  4),
    'best_val_auc': round(best_auc, 4),
    'test_acc':     round(test_acc, 4),
    'test_f1':      round(test_f1,  4),
    'test_auc':     round(test_auc, 4),
    'epochs_run':   len(history['epoch']),
    'history':      history,
}
with open(f'{OUTPUT}vgg16_result.json','w') as f:
    json.dump(result, f, indent=2)

torch.cuda.empty_cache()

print("="*55)
print("  ✓ VGG16 COMPLETE")
print("="*55)
print("  Download from Output panel:")
files = [f for f in os.listdir(OUTPUT) if 'vgg16' in f]
for fname in sorted(files):
    size = os.path.getsize(f'{OUTPUT}{fname}') / 1024
    unit = 'MB' if size > 1024 else 'KB'
    size = size/1024 if size > 1024 else size
    print(f"    {fname:<45} {size:.1f} {unit}")
print("\n  Next: Run ResNet50 notebook")
